In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# Datasets and Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms  as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10("./Dataset", train=True, download=True, transform=transform)
testset = CIFAR10("./Dataset", train=False, download=True, transform=transform)

100%|████████████████████████████████████████████████████████████████████████████████| 170M/170M [21:27<00:00, 132kB/s]


In [7]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [8]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Dataset
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

## Build the model

In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [19]:
model = CNN()

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the model

In [ ]:
epochs = 10
training_loss = []
validation_loss = []

for epoch in range(epochs):
    epoch_training_loss = 0.0
    epoch_validation_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model(images) # fp
        loss = criterion(output, labels) # loss fnx
        loss.backward() # bp
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    running_training_loss = epoch_training_loss/len(trainloader)
    training_loss.append(running_training_loss)

    model.eval()
    with torch.no_grad():
        for images, labels in trainloader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            epoch_validation_loss +=  loss.item()
            
    running_validation_loss = epoch_validation_loss/len(trainloader)
    validation_loss.append(running_validation_loss)

    print(f"epoch = {epoch+1}/{epochs} , training loss = {running_validation_loss}, validation loss = {running_training_loss}")

epoch = 1/10 , training loss = 0.0368738875564267, validation loss = 0.05064496585840682
epoch = 2/10 , training loss = 0.02340143664019948, validation loss = 0.048691184626763584


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

loss_df =  pd.DataFrame({
    "Training Loss": training_loss,
    "Validation Loss": validation_loss
})

plt.figure(figsize=(10,5))
plt.plot(loss_df["Training Loss"], label="Training Loss")
plt.plot(loss_df["Validation Loss"], label="Validation Loss", color="white")

plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.legend()


## Evalute

In [29]:
total_labels = 0
correct_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

    print(f"accuracy = {(correct_labels/total_labels)*100}")

accuracy = 75.71
